<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/Private_Credit_%26_Direct_Lending_Portfolio_Model_Risk_Validation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Executive Summary

Private credit (direct lending to middle-market companies with \$10M - \$100M EBITDA) differs fundamentally from traditional public high-yield or syndicated C&I bank loans:

1. **Unrated Middle-Market Obligors:** Private borrowers lack public credit ratings from Moody's / S&P. Risk models rely on Shadow Rating / Internal Rating engines mapped to financial ratios.
2. **Floating-Rate Exposure (SOFR Shift Sensitivity):** Private debt deals are almost exclusively floating-rate (SOFR + Spread). When base rates rise, borrower interest expenses spike instantly, severly compressing the **Debt Service Coverage Ratio (DSCR)** even if EBITDA remains flat.
3. **Covenant-Heavy Dynamics & Default Definitions:** Default  in private credit is rarely limited to bankruptcy or 90-day delinquency. It includes **Covenant Breaches** (e.g., Net Debt/EBITDA > Covenant Cap), **Payment-In-Kind (PIK) Toggle Triggers**, or **Equity Cures**.

# 2. Mathematical Framework


#### 1. Floating-Rate DSCR Squeeze & Interest Expenses Model
For obligor $i$ in quarter $t$, the floating interest rate is $r_{t} = \text{SOFR}_{t} + \text{Spread}_{i}$. The Debt Service Coverage Ratio ($DSCR$) is:


$$DSCR_{i,t} = \frac{\text{EBITDA}_{i,t} - \text{Capex}_{i,t} - \text{Taxes}_{i,t}}{(\text{SOFR}_{t} + \text{Spread}_{i}) \times \text{Total Debt}_{i,t} + \text{Mandatory Principal}_{i,t}}$$

If $DSCR_{i,t} < 1.0$, the borrower cannot cover cash debt service without drawing on revolver lines, triggering a PIK toggle, or requiring a sponsor equity cure.


#### 2. Shadow Rating Mapping (Multinomial / Ordinal Logit)
Borrowers are assigned a latent credt score $y^{*}_{i}$ based on leverage, coverage, and operational scale:

$$y^{*}_{i}=\beta_{0} + \beta_{1}ln(\text{EBITDA}_{i})+\beta_{2}\left(\frac{\text{Total Debt}_{i}}{\text{EBITDA}_{i}}\right)+\beta_{3}DSCR_{i}+\beta_{4}\Delta \text{EBITDA}_{i}+\epsilon_{i}$$

The probability of belonging to internal rating buc ket $k \in \{1,..., K\}$ (e.g., 1 = Low Risk / BB, $K$ = Impaired / CCC) is given by cutpoint $\theta_{k}$:

$$P(Y_{i} = k) = \Phi(\theta_{k} - y^{*}_{i}) - \Phi(\theta_{k-1} - y^{*}_{i})$$

#### 3. Dynamic Enterprise Value (EV) LGD Waterfall

Unlike fixed regulatory LGD assumptions (e.g., 45%), private credit LGD depends on total enterprise value relative to debt claims:

$$EV_{i,t} = \text{EBITDA}_{i,t} \times M_{EV,s}(t)$$

where $M_{EV,s}(t)$ is the prevailing sector-specific EV multiple under macro stress scenario $s$. For a Unitranche loan position $D_{Uni,i}$ with senior claim recovery:

$$LGD_{i,t}(\text{M}_{\text{EV},s}) = \max \left(0, 1 - \frac{\min(EV_{i,t}\times (1 - \text{Distress Haircut}), D_{\text{Uni},i})}{D_{\text{Uni},i}}\right)$$


In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.metrics import confusion_matrix, roc_auc_score

class PrivateCreditModelValidator:
  """
  Independent MRM Validation Engine for Private Credit & Direct Lending Portfolios.
  Evaluates SOFR Interest Rate Sensitivity, Covenant Breaches, EV LGD Waterfalls, and Shadow Ratings.
  """
  def __init__(self, portfolio_df: pd.DataFrame, master_rating_scale: dict):
    """
    portfolio_df requires columns:
    ['Obligor_ID', 'EBITDA', 'Total_Debt', 'SOFR_Spread', 'Covenant_Leverage_Cap',
    'Capex_Tax_Rate', 'EV_Multiple_Base', 'Seniority']
    """
    self.portfolio = portfolio_df.copy()
    self.rating_scale = master_rating_scale

  def stress_sofr_interest_coverage(self, sofr_base: float, sofr_shock: float) -> pd.DataFrame:
    """
    Simulates interest expense expanision and DSCR compression under SOFR rate spikes.
    Identifies borrowers crossing DSCR < 1.0 (Cash Flow Distress) or Leverage > Covenant Cap.
    """
    df = self.portfolio.copy()

    # Base and Shocked Interest Rates
    r_base = sofr_base + df['SOFR_Spread']
    r_shock = sofr_shock + df['SOFR_Spread']

    # Free Cash Flow before interest
    fcf_pre_interest = df['EBITDA'] * (1.0 - df['Capex_Tax_Rate'])

    # Base vs Shocked DSCR
    df['Interest_Exp_Base'] = df['Total_Debt'] * r_base
    df['Interest_Exp_Shock'] = df['Total_Debt'] * r_shock

    df['DSCR_Base'] = fcf_pre_interest / np.maximum(df['Interest_Exp_Base'], 1e-4)
    df['DSCR_Shock'] = fcf_pre_interest / np.maximum(df['Interest_Exp_Shock'], 1e-4)

    # Current Leverage vs Covenant Cap
    df['Current_Leverage'] = df['Total_Debt'] / np.maximum(df['EBITDA'], 1e-4)
    df['Covenant_Breach_Flag'] = df['Current_Leverage'] > df['Covenant_Leverage_Cap']
    df['DSCR_Distress_Flag'] = df['DSCR_Shock'] < 1.0

    return df

  def calculate_ev_dynamic_lgd(self, df: pd.DataFrame, ev_multiple_haircut: float = 0.25, liquidation_cost: float = 0.15) -> pd.DataFrame:
    """
    Calculates dynamic LGD based on Enterprise Value (EV) waterfall coverage under sector multiple sector compression.
    """
    results = df.copy()

    # Stressed EV = EBITDA * (Base Multiple * (1 - Multiple Haircut))
    stressed_ev_multiple = results['EV_Multiple_Base'] * (1.0 - ev_multiple_haircut)
    enterprise_value = results['EBITDA'] * stressed_ev_multiple

    # Net recoverable value after distress liquidation cost
    recoverable_ev = enterprise_value * (1.0 - liquidation_cost)

    # Recovery and LGD Calculation
    covered_debt = np.minimum(recoverable_ev, results['Total_Debt'])
    recovery_rate = covered_debt / np.maximum(results['Total_Debt'], 1e-4)

    results['Stressed_EV_Multiple'] = stressed_ev_multiple
    results['Enterprise_Value'] = enterprise_value
    results['Dynamic_LGD'] = np.clip(1.0 - recovery_rate, 0.0, 1.0)

    return results

  def shadow_rating_mapping(self, df: pd.DataFrame) -> pd.Series:
    """
    Maps continuous financial metrics to an internal Shadow Rating scale.
    """
    scores = []
    for _, row in df.iterrows():
      lev = row['Current_Leverage']
      dscr = row['DSCR_Shock']
      ebitda = row['EBITDA']

      # Latent Score Formula
      z = 0.6 * lev - 1.2 * dscr - 0.2 * np.log(ebitda)

      if z < -1.0:
        rating = "B1_LowRisk"
      elif z < 0.5:
        rating = "B2_ModerateRisk"
      elif z < 2.0:
        rating = "B3_HighRisk"
      else:
        rating = "CCC_Impaired"
      scores.append(rating)

    return pd.Series(scores, index=df.index)

  def run_portfolio_stress_test(self, sofr_base: float, sofr_shock: float, ev_haircut: float) -> dict:
    """
    Executes full MRM stress validation pipeline across rates, covenants, and LGD.
    """
    # Step 1: Stress Interest Coverage & Covenants
    stressed_df = self.stress_sofr_interest_coverage(sofr_base, sofr_shock)

    # Step 2: Compute Dynamic EV LGD
    full_df = self.calculate_ev_dynamic_lgd(stressed_df, ev_multiple_haircut=ev_haircut)

    # Step 3: Assign Shadow Ratings
    full_df['Shadow_Rating'] = self.shadow_rating_mapping(full_df)

    # Summary Metrics
    total_exposure = full_df['Total_Debt'].sum()
    covenant_breach_exposure = full_df[full_df['Covenant_Breach_Flag']]['Total_Debt'].sum()
    dscr_distress_exposure = full_df[full_df['DSCR_Distress_Flag']]['Total_Debt'].sum()

    # Calculate Expected Loss ($) =  Exposure * (PD_proxy) * LGD
    # Using simple PD proxy mapping based on shadow rating
    pd_map = {"B1_LowRisk": 0.015, "B2_ModerateRisk": 0.04, "B3_HighRisk": 0.10, "CCC_Impaired": 0.28}
    full_df['PD_Proxy'] = full_df['Shadow_Rating'].map(pd_map)
    full_df['Expected_Loss_Dollar'] = full_df['Total_Debt'] * full_df['PD_Proxy'] * full_df['Dynamic_LGD']

    total_el = full_df['Expected_Loss_Dollar'].sum()

    return {
        "Total_Portfolio_Exposure": total_exposure,
        "Covenant_Breach_Pct": float((covenant_breach_exposure / total_exposure) * 100),
        "DSCR_Distress_Pct": float((dscr_distress_exposure / total_exposure) * 100),
        "Weighted_Average_LGD": float(np.average(full_df['Dynamic_LGD'], weights=full_df['Total_Debt']) * 100),
        "Total_Expected_Loss_Dollar": float(total_el),
        "Portfolio_EL_Rate": float((total_el / total_exposure) * 100),
        "Detailed_Portfolio": full_df
    }

# ---- Example Usage Demonstration ----
if __name__ == "__main__":
  np.random.seed(42)
  n_loans = 100

  # Simulate Middle Market Private Credit Portfolio ($1.5B Total Assets)
  portfolio_data = pd.DataFrame({
      'Obligor_ID': [f"MM_Loan_{i:03d}" for i in range(1, n_loans + 1)],
      'EBITDA': np.random.uniform(8.0, 45.0, n_loans),                  # $8M - $45M EBITDA
      'Total_Debt': np.random.uniform(30.0, 200.0, n_loans),            # $30M - $200M Debt
      'SOFR_Spread': np.random.uniform(0.055, 0.075, n_loans),          # S + 550bps to 750bps
      'Covenant_Leverage_Cap': np.random.uniform(5.0, 6.5, n_loans),    # Leverage cap 5.0x - 6.5x
      'Capex_Tax_Rate': np.random.uniform(0.15, 0.25, n_loans),         # FCF drain
      'EV_Multiple_Base': np.random.uniform(8.5, 12.5, n_loans),        # Entry EV multples
      'Seniority': ['Unitranche'] * n_loans
  })

  master_scale = {"B1": 0.015, "B2": 0.04, "B3": 0.10, "CCC": 0.28}

  validator = PrivateCreditModelValidator(portfolio_data, master_scale)

  # Run Stress Test: SOFR spikes from 2.5% to 5.5% (+300bps) and EV Multiples compress by 20%
  report = validator.run_portfolio_stress_test(sofr_base=0.025, sofr_shock=0.055, ev_haircut=0.20)

  print("=== Private Credit Portfolio Model Validation Report ===")
  print(f"Total Portfolio Exposure: ${report['Total_Portfolio_Exposure']:,.2f}M")
  print(f"Covenant Breach Exposure: {report['Covenant_Breach_Pct']:.2f}%")
  print(f"DSCR < 1.0 Cash Distress Exposure: {report['DSCR_Distress_Pct']:.2f}%")
  print(f"Portfolio Weighted Stressed LGD: {report['Weighted_Average_LGD']:,.2f}%")
  print(f"Stressed Expected Loss ($): ${report['Total_Expected_Loss_Dollar']:,.2f}M")
  print(f"Portfolio Loss Rate: {report['Portfolio_EL_Rate']:.2f}%")

  print("\n === Sample Obligor Level Validation Results ===")
  sample_out = report['Detailed_Portfolio'][['Obligor_ID', 'EBITDA', 'Current_Leverage', 'DSCR_Shock', 'Dynamic_LGD', 'Shadow_Rating']].head(5)
  print(sample_out.to_string(index=False))

=== Private Credit Portfolio Model Validation Report ===
Total Portfolio Exposure: $11,463.14M
Covenant Breach Exposure: 45.53%
DSCR < 1.0 Cash Distress Exposure: 42.31%
Portfolio Weighted Stressed LGD: 13.06%
Stressed Expected Loss ($): $419.27M
Portfolio Loss Rate: 3.66%

 === Sample Obligor Level Validation Results ===
 Obligor_ID    EBITDA  Current_Leverage  DSCR_Shock  Dynamic_LGD   Shadow_Rating
MM_Loan_001 21.857984          1.616936    4.227493     0.000000      B1_LowRisk
MM_Loan_002 43.176429          3.200584    2.125457     0.000000      B1_LowRisk
MM_Loan_003 35.083776          2.378322    2.968675     0.000000      B1_LowRisk
MM_Loan_004 30.150364          3.862541    1.552426     0.000000 B2_ModerateRisk
MM_Loan_005 13.772690         13.380560    0.500563     0.428838    CCC_Impaired
